# The three-path tranche

After the simulation, a stateless calculator amortizes every tranche into monthly expense
(`hcsim.financial.expense`). This notebook takes one tranche down three paths and works its
expense out by hand. Its holder stays to the vest, leaves before it, or moves to another cost
center and then leaves. Every number here is invented for the example.

The tranche is 120 shares, awarded in month 0 to vest in month 5, valued at 10 a share: 1,200.
The flat forfeiture rate expects 10% of shares to forfeit. Without a calendar origin, each month
from the award through the vest accrues an equal share of the value. A run gives the origin, and
then each month's share is in proportion to its calendar days.

In [ ]:
import numpy as np

from hcsim.financial.expense import ExpenseConfig, calculate, prepare
from hcsim.financial.tracker import HolderLedger

ledger = HolderLedger()
schedule = np.zeros((3, 72), dtype=np.int64)
schedule[:, 4] = 120  # column c vests c + 1 months after the award: month 5
ledger.record(np.array([1, 2, 3]), 0, schedule, 0, np.zeros(3, int), np.zeros(3, int))
ledger.forfeit(np.array([2]), 3)  # the second holder leaves in month 3
ledger.move(np.array([3]), 2, np.array([1]))  # the third moves to CC-02 in month 2
ledger.forfeit(np.array([3]), 4)  # and leaves in month 4

prepared = prepare(ledger.tranches(), np.full(3, 10.0), intervals=ledger.intervals())
result = calculate(prepared, ExpenseConfig(forfeiture_rate=0.10))
rows = result.rows
rows["recognized"] = rows[["net", "trueup_attrition", "trueup_rate", "trueup_vesting"]].sum(axis=1)
rows

## Accrual

The six months 0 to 5 each accrue 1,200 / 6 = 200 gross. Net of the 10% expected to forfeit,
that's 180 a month. A month accrues if the holder still holds the tranche when it starts, so the
month a holder leaves accrues too.

In [ ]:
accruing = rows[rows["gross"] > 0]
assert np.allclose(accruing["gross"], 200)
assert np.allclose(accruing["net"], 180)

## Path 1: the holder stays, and the tranche vests

Six months accrue 6 x 180 = 1,080. At the vest, the **vesting true-up** books the residual
between the value and what was recognized: 1,200 - 1,080 = 120, the 10% that was expected to
forfeit but didn't. The tranche recognizes exactly its vested value.

In [ ]:
stays = rows[rows["tranche"] == 0]
assert np.allclose(stays["trueup_vesting"], [0, 0, 0, 0, 0, 120])
assert np.isclose(stays["recognized"].sum(), 1_200)

## Path 2: the holder leaves in month 3

Months 0 to 3 accrue 4 x 180 = 720. Leaving forfeits the tranche, and the **attrition true-up**
reverses everything it recognized, so it recognizes nothing: no shares vested.

In [ ]:
leaves = rows[rows["tranche"] == 1]
assert leaves["month"].tolist() == [0, 1, 2, 3]
assert np.allclose(leaves["trueup_attrition"], [0, 0, 0, -720])
assert np.isclose(leaves["recognized"].sum(), 0)

## Path 3: the holder moves in month 2, then leaves in month 4

Each month is booked in the cost center the holder sat in that month: months 0 and 1 in CC-01,
which accrue 360, and months 2 to 4 in CC-02, which accrue 540. When the holder leaves, the
attrition true-up takes the **forfeited-piece path**. It reverses each piece where it was
booked, so each cost center's books net to nothing.

In [ ]:
moves = rows[rows["tranche"] == 2]
books = moves.groupby("cost_center")[["net", "trueup_attrition"]].sum()
assert np.allclose(books.loc[0], [360, -360])  # CC-01
assert np.allclose(books.loc[1], [540, -540])  # CC-02
books

## The identity

On every path, cumulative net expense plus the true-ups equals the vested shares times the
value per share: 1,200 on the first path, nothing on the others. The calculator checks it for
every tranche, and a residual above 0.01 would mean a defect. A flat rate never changes, so the
**rate true-up** stays at zero here. With per-cohort rates, it adjusts a still-accruing
tranche's earlier accrual whenever its cohort's rate moves.

In [ ]:
assert np.abs(result.residual).max() < 0.01
assert (rows["trueup_rate"] == 0).all()